<a href="https://colab.research.google.com/github/jacknzheng/pretraining/blob/main/tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."

tokens = text.encode("utf-8")
tokens = list(map(int, tokens))

In [ ]:

def get_stats(ids):
  counts = {}
  for pair in zip(ids, ids[1:]):
    counts[pair] = counts.get(pair, 0) + 1
  return counts

stats = get_stats(tokens)
print(sorted(((v,k) for k,v in stats.items()), reverse=True))

def merge(ids, pair, idx):
  newids = []
  i = 0
  while i < len(ids):

    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      newids.append(idx) # found the pair, add the new pair to new-ids
      i += 2
    else: # just append the id
      newids.append(ids[i])
      i += 1
  return newids

[(20, (101, 32)), (15, (240, 159)), (12, (226, 128)), (12, (105, 110)), (10, (115, 32)), (10, (97, 110)), (10, (32, 97)), (9, (32, 116)), (8, (116, 104)), (7, (159, 135)), (7, (159, 133)), (7, (97, 114)), (6, (239, 189)), (6, (140, 240)), (6, (128, 140)), (6, (116, 32)), (6, (114, 32)), (6, (111, 114)), (6, (110, 103)), (6, (110, 100)), (6, (109, 101)), (6, (104, 101)), (6, (101, 114)), (6, (32, 105)), (5, (117, 115)), (5, (115, 116)), (5, (110, 32)), (5, (100, 101)), (5, (44, 32)), (5, (32, 115)), (4, (116, 105)), (4, (116, 101)), (4, (115, 44)), (4, (114, 105)), (4, (111, 117)), (4, (111, 100)), (4, (110, 116)), (4, (110, 105)), (4, (105, 99)), (4, (104, 97)), (4, (103, 32)), (4, (101, 97)), (4, (100, 32)), (4, (99, 111)), (4, (97, 109)), (4, (85, 110)), (4, (32, 119)), (4, (32, 111)), (4, (32, 102)), (4, (32, 85)), (3, (118, 101)), (3, (116, 115)), (3, (116, 114)), (3, (116, 111)), (3, (114, 116)), (3, (114, 115)), (3, (114, 101)), (3, (111, 102)), (3, (111, 32)), (3, (108, 108)), (

In [ ]:
vocab_size = 276
num_merges = vocab_size - 256
ids = list(tokens) # all the UTF encoded tokens

merges = {} # (int, int) -> int

for i in range(num_merges):
  stats = get_stats(ids)
  pair = max(stats, key=stats.get)
  idx = 256 + i
  print(f"merged {pair} into new token {idx}")
  ids = merge(ids, pair, idx)
  merges[pair] = idx

merged (101, 32) into new token 256
merged (240, 159) into new token 257
merged (226, 128) into new token 258
merged (105, 110) into new token 259
merged (115, 32) into new token 260
merged (97, 110) into new token 261
merged (116, 104) into new token 262
merged (257, 133) into new token 263
merged (257, 135) into new token 264
merged (97, 114) into new token 265
merged (239, 189) into new token 266
merged (258, 140) into new token 267
merged (267, 264) into new token 268
merged (101, 114) into new token 269
merged (111, 114) into new token 270
merged (116, 32) into new token 271
merged (259, 103) into new token 272
merged (115, 116) into new token 273
merged (261, 100) into new token 274
merged (32, 262) into new token 275


In [ ]:
# dictionary of number: byte encoding
vocab = {idx: bytes([idx]) for idx in range(256)}

for (p0,p1), idx in merges.items(): # (101, 32), 257
  vocab[idx] = vocab[p0] + vocab[p1] # index 257 = (101, 32)

def decode(ids):
  # convert encoding to UTF-8 using vocab[idx], then join into a long string
  tokens = b"".join(vocab[idx] for idx in ids)

  # not all byte sequences are valid utf-8
  text = tokens.decode("utf-8", errors="replace")
  return text

print(decode([128]))

�


In [ ]:
merges

{(101, 32): 256,
 (240, 159): 257,
 (226, 128): 258,
 (105, 110): 259,
 (115, 32): 260,
 (97, 110): 261,
 (116, 104): 262,
 (257, 133): 263,
 (257, 135): 264,
 (97, 114): 265,
 (239, 189): 266,
 (258, 140): 267,
 (267, 264): 268,
 (101, 114): 269,
 (111, 114): 270,
 (116, 32): 271,
 (259, 103): 272,
 (115, 116): 273,
 (261, 100): 274,
 (32, 262): 275}

In [ ]:
final = []

def encode_jack(text):
  encode = list(text.encode("utf-8"))
  for (p0,p1), idx in merges:
    for i in range(len(encode)):
      if encode[i] == p0 and encode[i+1] == p1:
        # replace
        final.append(idx)
      else:
        final.append(encode[i])
  return final


def encode(text):
  tokens = list(text.encode("utf-8"))
  while len(tokens) >= 2:
    stats = get_stats(tokens) # zip all the pairs giving (p0, p1), idx

    # for each pair in stats, find if it exists in merges array
    pair = min(stats, key=lambda p: merges.get(p, float("inf")))
    if pair not in merges: # no more adjacent pairs in tokens, that match merges
      break
    idx = merges[pair]
    tokens = merge(tokens, pair, idx)
  return tokens

print(decode(encode("hello world")))

hello world


Implementing better BPE

https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

In [ ]:
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Hello world"))

['Hello', ' world']


In [ ]:
import tiktoken

# GPT-2 (does not merge spaces)
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    hello world!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    hello world!!!"))

[220, 220, 220, 23748, 995, 10185]
[262, 24748, 1917, 12340]


In [ ]:
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/vocab.bpe
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/encoder.json

--2026-03-15 01:22:11--  https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/vocab.bpe
Resolving openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)... 20.60.244.1
Connecting to openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)|20.60.244.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 456318 (446K) [application/octet-stream]
Saving to: ‘vocab.bpe’

vocab.bpe           100%[===================>] 445.62K  1.71MB/s    in 0.3s    

2026-03-15 01:22:12 (1.71 MB/s) - ‘vocab.bpe’ saved [456318/456318]

--2026-03-15 01:22:12--  https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/encoder.json
Resolving openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)... 20.60.244.1
Connecting to openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)|20.60.244.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1042301 (1018K) [application/json]
Saving to: ‘enc

In [ ]:
import os, json

with open('encoder.json', 'r') as f:
    encoder = json.load(f) # <--- ~equivalent to our "vocab"

with open('vocab.bpe', 'r', encoding="utf-8") as f:
    bpe_data = f.read()
bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split('\n')[1:-1]]
# ^---- ~equivalent to our "merges"


## **sentencepiece**

In [ ]:
import sentencepiece as spm